In [1]:
import pickle

# Load collaborative embedding
with open("../../data/games_with_embeddings.pkl", "rb") as f:
    game_embedding_dict = pickle.load(f)

In [2]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def find_closest_games(target_game, embeddings_dict, top_n=5):
    """
    Finds the most similar games based on their embedding vectors.
    
    :param target_game: Name of the game to search for (str)
    :param embeddings_dict: Dictionary mapping Game Name -> Numpy Array (size 32)
    :param top_n: Number of recommendations to return
    """
    
    # Check if the game exists in our database
    if target_game not in embeddings_dict:
        return f"Error: '{target_game}' not found in the embeddings database."

    # Isolate the target vector and reshape it for sklearn
    # reshape(1, -1) turns it from shape (32,) to (1, 32)
    target_vector = embeddings_dict[target_game].reshape(1, -1)
    
    # Prepare the rest of the data
    # We separate names and vectors so their indexes match up
    game_names = list(embeddings_dict.keys())
    all_vectors = np.array(list(embeddings_dict.values()))
    
    # Calculate Cosine Similarity
    # This compares the target (1, 32) against all games (N, 32) simultaneously
    # It returns an array of scores from -1.0 (opposites) to 1.0 (identical)
    similarity_scores = cosine_similarity(target_vector, all_vectors)[0]
    
    # Sort the results
    # argsort() gives us the indexes from lowest to highest, so we reverse it [::-1]
    ranked_indexes = np.argsort(similarity_scores)[::-1]
    
    # Format the output
    print(f"Games most similar to '{target_game}':\n")
    results = []
    
    for idx in ranked_indexes:
        match_name = game_names[idx]
        score = similarity_scores[idx]
        
        # Skip the target game itself (it will always have a 1.0 score)
        if match_name == target_game:
            continue
            
        results.append((match_name, score))
        print(f"{len(results)}. {match_name} (Similarity Score: {score:.4f})")
        
        # Stop once we hit our desired number of recommendations
        if len(results) == top_n:
            break
            
    return results

In [ ]:
# For reproducible random numbers
np.random.seed(42) 

# Run the function
find_closest_games("Marvel's Spider-Man 2", game_embedding_dict, top_n=10)

Games most similar to 'Black Myth: Wukong':

1. Metal Gear Solid Delta: Snake Eater (Similarity Score: 0.9888)
2. TankHead (Similarity Score: 0.9887)
3. Unless (Similarity Score: 0.9879)
4. Nevermind This (Similarity Score: 0.9879)
5. Toby's Topsy Tale (Similarity Score: 0.9879)
6. Hunted Within: The Walls (Similarity Score: 0.9879)
7. Stolen Rage (Similarity Score: 0.9879)
8. A Twisted Tale (Similarity Score: 0.9879)
9. A prisioneira da Noite - The prisoner of the Night (Similarity Score: 0.9879)
10. Harmony Of Fear (Similarity Score: 0.9879)


[('Metal Gear Solid Delta: Snake Eater', np.float64(0.9888358163452398)),
 ('TankHead', np.float64(0.988676660130021)),
 ('Unless', np.float64(0.9878790482727464)),
 ('Nevermind This', np.float64(0.9878790482727464)),
 ("Toby's Topsy Tale", np.float64(0.9878790482727464)),
 ('Hunted Within: The Walls', np.float64(0.9878790482727464)),
 ('Stolen Rage', np.float64(0.9878790482727464)),
 ('A Twisted Tale', np.float64(0.9878790482727464)),
 ('A prisioneira da Noite - The prisoner of the Night',
  np.float64(0.9878790482727464)),
 ('Harmony Of Fear', np.float64(0.9878790482727464))]